In [1]:
# Cell 1: Clone repo + install deps
%cd /content
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
%cd /content/cibuco-boriken
print("Setup complete")

/content
Cloning into 'cibuco-boriken'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 159 (delta 80), reused 107 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 273.53 KiB | 3.04 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/cibuco-boriken
Setup complete


In [2]:
!pip install -q tensorflow tensorflow-hub librosa



In [3]:
# Cell 2: Kaggle credentials (secure)
import os
import json
from google.colab import userdata

# Store token in Colab Secrets (never in code)
# Steps:
#   1. Click the 🔑 key icon in left sidebar
#   2. Add secret name: KAGGLE_USERNAME → drosadocastro-bit
#   3. Add secret name: KAGGLE_KEY → your_new_token
#   4. Enable notebook access for both

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_creds = {
    "username": userdata.get('kaggle_username'),
    "key": userdata.get('kaggle_key')
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle configured securely ✅")

# Download data
!pip install -q kaggle==1.6.17
!kaggle competitions download -c birdclef-2026
!mkdir -p data/birdclef-2026
!unzip -q birdclef-2026.zip -d data/birdclef-2026
print("Data ready ✅")


Kaggle configured securely ✅
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
100% 14.9G/15.0G [01:10<00:00, 290MB/s]
100% 15.0G/15.0G [01:10<00:00, 229MB/s]
Data ready ✅


In [9]:
# Cell 3: Mount Drive (save model after training)
import os
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!python -m birdclef.perch_embed --save-model --model-dir perch_saved_model

2026-03-17 14:20:59.915533: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 14:20:59.923732: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773757259.932995    7341 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773757259.935986    7341 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773757259.943536    7341 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [11]:
import os

# Set BEFORE any birdclef imports
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

# Force reload entire birdclef module
import importlib
import birdclef
import birdclef.config as cfg

importlib.reload(cfg)

# Manually override if reload didn't catch it
from pathlib import Path
cfg.COMPETITION_DIR = Path(os.environ['BIRDCLEF_DATA_DIR'])
cfg.TRAIN_AUDIO_DIR = cfg.COMPETITION_DIR / "train_audio"
cfg.TEST_AUDIO_DIR  = cfg.COMPETITION_DIR / "test_soundscapes"
cfg.TRAIN_META_CSV  = cfg.COMPETITION_DIR / "train.csv"
cfg.TAXONOMY_CSV    = cfg.COMPETITION_DIR / "taxonomy.csv"
cfg.SAMPLE_SUBMISSION = cfg.COMPETITION_DIR / "sample_submission.csv"

# Verify
print(f"Data dir: {cfg.COMPETITION_DIR}")
print(f"Train CSV exists: {cfg.TRAIN_META_CSV.exists()}")
print(f"Train audio exists: {cfg.TRAIN_AUDIO_DIR.exists()}")
print(f"Soundscapes exists: {cfg.COMPETITION_DIR / 'train_soundscapes_labels.csv'}")

Data dir: /content/cibuco-boriken/data/birdclef-2026
Train CSV exists: True
Train audio exists: True
Soundscapes exists: /content/cibuco-boriken/data/birdclef-2026/train_soundscapes_labels.csv


In [13]:
import os
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

# Verify env var is set
print(os.environ.get('BIRDCLEF_DATA_DIR'))

# Check file directly without config
from pathlib import Path
p = Path('/content/cibuco-boriken/data/birdclef-2026/train.csv')
print(f"File exists directly: {p.exists()}")

/content/cibuco-boriken/data/birdclef-2026
File exists directly: True


In [20]:
# Pull the fix
%cd /content/cibuco-boriken
!git pull origin main

# Re-run extraction
!python -m birdclef.perch_embed --extract \
    --model-dir perch_saved_model \
    --train-csv /content/cibuco-boriken/data/birdclef-2026/train.csv \
    --audio-dir /content/cibuco-boriken/data/birdclef-2026/train_audio \
    --output-dir perch_embeddings

/content/cibuco-boriken
From https://github.com/drosadocastro-bit/cibuco-boriken
 * branch            main       -> FETCH_HEAD
Already up to date.
2026-03-17 14:31:08.369591: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 14:31:08.377858: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773757868.387223   11287 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773757868.390275   11287 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registe

In [21]:
!python -m birdclef.perch_classify \
    --embeddings-dir perch_embeddings \
    --epochs 50 \
    --patience 10 \
    --output perch_head.pt

birdclef.perch_classify | INFO | Loaded 118956 embeddings, 206 species
birdclef.perch_classify | INFO | Training PerchHead: 50 epochs, batch=256, lr=0.001
birdclef.perch_classify | INFO | Epoch   1/50 | Train Loss: 0.162739 | Val Loss: 0.032964 | Time: 1.3s
birdclef.perch_classify | INFO |   -> Saved best (val_loss=0.032964)
birdclef.perch_classify | INFO | Epoch   2/50 | Train Loss: 0.035327 | Val Loss: 0.032458 | Time: 1.1s
birdclef.perch_classify | INFO |   -> Saved best (val_loss=0.032458)
birdclef.perch_classify | INFO | Epoch   3/50 | Train Loss: 0.034898 | Val Loss: 0.032370 | Time: 0.9s
birdclef.perch_classify | INFO |   -> Saved best (val_loss=0.032370)
birdclef.perch_classify | INFO | Epoch   4/50 | Train Loss: 0.034709 | Val Loss: 0.033583 | Time: 0.9s
birdclef.perch_classify | INFO | Epoch   5/50 | Train Loss: 0.034472 | Val Loss: 0.031095 | Time: 0.9s
birdclef.perch_classify | INFO |   -> Saved best (val_loss=0.031095)
birdclef.perch_classify | INFO | Epoch   6/50 | Train 

In [26]:
# SAVE MODEL IMMEDIATELY AFTER TRAINING
# Save to Drive
import shutil
shutil.copytree('perch_saved_model', '/content/drive/MyDrive/perch_saved_model', dirs_exist_ok=True)
shutil.copy('perch_head.pt', '/content/drive/MyDrive/perch_head.pt')
print('Saved to Drive ✅'),


Saved to Drive ✅


(None,)

In [ ]:
# Cell 5: CFAR Phase 2 k-sweep (3 conditions)
import os
os.environ['BIRDCLEF_DATA_DIR'] = '/content/cibuco-boriken/data/birdclef-2026'

!BIRDCLEF_DATA_DIR=/content/cibuco-boriken/data/birdclef-2026 \
  python -m birdclef.evaluate_thresholds \
  --backbone efficientnet_b2 \
  --include-soundscapes \
  --k-sweep 1.0 2.0 3.0 \
  --temperature 0.3

print("k-sweep complete ✅")

In [ ]:
# Cell 6: Display and save figure/model
from IPython.display import Image, display
import shutil

display(Image('k_sweep_figure.png'))

shutil.copy('k_sweep_figure.png', SAVE_DIR + 'k_sweep_500samples.png')
shutil.copy('birdclef/models/birdclef_model.pt', SAVE_DIR + 'birdclef_model_500samples.pt')
print("Saved to Drive")

In [ ]:
# Cell 7: Print paper-ready results table
import json
from pathlib import Path

results_path = Path('k_sweep_results.json')
if not results_path.exists():
    print('k_sweep_results.json not found. Run Cell 5 first.')
else:
    rows = json.loads(results_path.read_text(encoding='utf-8'))
    print('## Table 1: CFAR k-Sensitivity Results (500 samples)')
    print('| k | F1 Fixed | F1 CFAR | FPR Fixed | FPR CFAR | T_mean |')
    print('|---|----------|---------|-----------|----------|--------|')
    for r in rows:
        print(f"| {r['k']:.1f} | {r['f1_fixed']:.4f} | {r['f1_cfar']:.4f} | {r['fpr_fixed']:.4f} | {r['fpr_cfar']:.4f} | {r['threshold_mean']:.4f} |")

In [ ]:
# Temperature Goldilocks curve
import matplotlib.pyplot as plt
import numpy as np

temps  = [0.05,   0.10,   0.20,   0.30,   0.50]
deltas = [0.0001, 0.0003, 0.0012, 0.0019, -0.0394]
aucs   = [0.5934, 0.7525, 0.7582, 0.7582, 0.7582]
fprs   = [0.0003, 0.0003, 0.0004, 0.0006, 0.0011]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 2: Temperature Scaling — Goldilocks Zone for CFAR')

# Left: F1 delta vs temperature
ax1.plot(temps, deltas, 'b-o', linewidth=2, markersize=8)
ax1.axhline(y=0, color='gray', linestyle='--', label='Fixed baseline')
ax1.axvline(x=0.3, color='green', linestyle='--', alpha=0.7, label='Optimal T=0.3')
ax1.fill_between(temps, deltas, 0,
                  where=[d>0 for d in deltas],
                  alpha=0.2, color='green', label='CFAR advantage')
ax1.fill_between(temps, deltas, 0,
                  where=[d<0 for d in deltas],
                  alpha=0.2, color='red', label='CFAR disadvantage')
ax1.set_xlabel('Temperature T')
ax1.set_ylabel('F1 Delta (CFAR - Fixed)')
ax1.set_title('F1 Improvement vs Temperature')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: AUC vs temperature
ax2.plot(temps, aucs, 'r-s', linewidth=2, markersize=8)
ax2.axvline(x=0.3, color='green', linestyle='--', alpha=0.7, label='Optimal T=0.3')
ax2.set_xlabel('Temperature T')
ax2.set_ylabel('ROC-AUC')
ax2.set_title('AUC vs Temperature')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('temperature_goldilocks.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved ✅")